# Phonon dispersion and DOS (DFPT)

`get_phonon_dispersion()` and `get_phonon_dos()` compute phonons via density
functional perturbation theory (DFPT, Elk tasks 205 then 220/210 --
`src/elkpy/calculation.py`). DFPT cost is dominated by the number of atomic-
displacement perturbations times q-points, not by anything elkpy controls.

**This notebook's cells are intentionally left unexecuted.** Even at the smallest
meaningful grid on a 2-atom cell (`ngridq=(2, 2, 2)`), a single call took ~11-13
minutes on the machine `tests/test_calculation_si_phonons.py` was last verified on
(that test suite is itself skipped by default -- set `ELKPY_RUN_SLOW_TESTS=1` to run
it). Running both cells below back-to-back is a ~25 minute wait, which isn't a good
fit for "run all cells" in a demo notebook; the code is real and has been verified
against a compiled Elk binary in `tests/test_calculation_si_phonons.py`, just not
re-run here. To execute it yourself:

```bash
./scripts/build_elk.sh   # if not already built
jupyter nbconvert --to notebook --execute --inplace notebooks/03_phonon_dispersion_and_dos.ipynb
```

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from elkpy.structure import Structure

SI_AVEC = [(5.13, 5.13, 0.00), (5.13, 0.00, 5.13), (0.00, 5.13, 5.13)]
SI_SPECIES = {"Si": [(0.0, 0.0, 0.0), (0.25, 0.25, 0.25)]}

structure = Structure(SI_AVEC, SI_SPECIES)
calc = structure.get_calculation(
    "_scratch/si", xc="PW", ngridk=(2, 2, 2), vkloff=(0.25, 0.5, 0.625)
)

## `get_phonon_dispersion()`

Same `vertices`/`kpath` interface as `get_bands()`. 2 atoms per cell -> 6 phonon
branches; the 3 acoustic branches go to ~0 frequency at Gamma.

In [ ]:
distances, frequencies = calc.get_phonon_dispersion(
    vertices=[(0.0, 0.0, 0.0), (0.5, 0.0, 0.0)], ngridq=(2, 2, 2), npoints=50
)
print("frequencies shape (nbranches, npoints):", frequencies.shape)

fig, ax = plt.subplots(figsize=(6, 4))
for branch in frequencies:
    ax.plot(distances, branch)
ax.set_xlabel("k-path distance")
ax.set_ylabel("frequency (Hartree)")
ax.set_title("Si phonon dispersion (DFPT, ngridq=2x2x2)")
fig.tight_layout()

## `get_phonon_dos()`

In [ ]:
frequencies_dos, phdos = calc.get_phonon_dos(ngridq=(2, 2, 2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(frequencies_dos, phdos)
ax.set_xlabel("frequency (Hartree)")
ax.set_ylabel("phonon DOS")
ax.set_title("Si phonon DOS (DFPT, ngridq=2x2x2)")
fig.tight_layout()

## Next

`04_per_species_soc_scaling.ipynb`, `05_berry_curvature.ipynb`,
`06_eigenstate_session.ipynb` -- elkpy's own Fortran extensions, all fast enough to
run live in their notebooks.